In [9]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
import pandas as pd
import os
import uuid
import shutil

In [10]:
os.environ["KEDRO_PACKAGE_NAME"] = "crispy_kedro"

workspace_dir = Path("workspace/results_v3")

tags=[
    "altrisk",
    # "reporting"
    ]


In [11]:
assets = pd.read_csv("data/05_model_input/downloaded_assets.csv")
companies = pd.read_csv("data/05_model_input/downloaded_companies.csv")

assets_companies = pd.merge(
    assets, 
    companies, 
    on=["asset_id", "sector", "technology", "production_year"],
    how="left"
)

[09/18/25 13:11:33] WARNING  /var/folders/df/zghzv05d7xb8t9xy7y5ld_h40000gn/T/ipykernel_74564/35376 ]8;id=386485;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py\warnings.py]8;;\:]8;id=716341;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py#109\109]8;;\
                             70851.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype                
                             option on import or set low_memory=False.                                             
                               assets = pd.read_csv("data/05_model_input/downloaded_assets.csv")                   
                                                                                                                   

In [ ]:
# Prepare companies summary to select a subset of companies later down

assets_companies_filtered = assets_companies[assets_companies["ownership_type"] == "direct"]
assets_companies_filtered = assets_companies_filtered[assets_companies_filtered["production_year"] == 2025]
assets_companies_filtered["capacity_owned"] = assets_companies_filtered["capacity"] * assets_companies_filtered["ownership_percentage"]
companies_summary = assets_companies_filtered.groupby(
    ["company_id", "company_name", "sector"], as_index=False
).agg(
    capacity_owned=("capacity_owned", "sum"),
    n_assets=("asset_id", "nunique"),
    n_countries=("country_iso2", "nunique")
).assign(
    capacity_owned=lambda x: x["capacity_owned"].astype(int),
)
companies_summary=  companies_summary.sort_values(by=["n_assets", "capacity_owned"], ascending=False).reset_index(drop=True)
companies_summary

,company_id,company_name,sector,capacity_owned,n_assets,n_countries
0,CP_7876088876044165226,other,Power,11464,358,31
1,CN_6660639238798673502,ENGIE SA,Power,17455,282,28
2,CN_5719632086864744401,EDF Renewables,Power,26700,279,26
3,CN_6785681074630732492,NextEra Energy Inc,Power,49174,256,3
4,CP_2362082173041183890,Iberdrola Renovables Energia SA,Power,8273,204,7
...,...,...,...,...,...,...
60536,CP_98321372751381949,Bem Querer hydroelectric plant,Power,0,1,1
60537,CP_989938990465740664,Shanxi Yangcheng Huangcheng Xiangfu Group Shis...,Coal,0,1,1
60538,CP_993937845921502505,Shanxi Qinyuan Guodao Jinyang Coal Industry Co...,Coal,0,1,1
60539,CP_996377999715439028,Balayan Bay Wind Power Project,Power,0,1,1


In [3]:
# Since the notebook is in ./notebooks, set the project path to the parent directory
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    os.chdir(current_dir.parent)
    print(f"Changed directory from {current_dir} to {Path.cwd()}")
else:
    print(f"Already in correct directory: {current_dir}")

metadata = bootstrap_project(project_path=Path.cwd())

Changed directory from /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/notebooks to /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro


[09/18/25 11:39:35] WARNING  /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/ ]8;id=892041;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py\warnings.py]8;;\:]8;id=178443;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py#109\109]8;;\
                             python3.10/site-packages/kedro/io/data_catalog.py:165:                                
                             KedroDeprecationWarning: `DataCatalog` has been deprecated and will be                
                             replaced by `KedroDataCatalog`, in Kedro 1.0.0.Currently some                         
                             `KedroDataCatalog` APIs have been retained for compatibility with                     
                             `DataCatalog`, including the `datasets` property and the                              
                             `get_datasets`, `_get_datasets`, `add`,` list`, `add_feed_dict`, and                  
                             `shallow_copy` methods. These will be removed or replaced with updated                
                             alternatives in Kedro 1.0.0. For more details, refer to the                           
                             documentation:                                                                        
                             https://docs.kedro.org/en/stable/data/index.html#kedrodatacatalog-expe                
                             rimental-feature                                                                      
                               warnings.warn(                                                                      
                                                                                                                   

In [4]:

workspace_dir.mkdir(parents=True, exist_ok=True)
print(f"Created workspace directory: {workspace_dir}")


Created workspace directory: workspace/results_v3


In [5]:
 
# import logging

# # quiet down Kedro loggers
# for name in [
#     "kedro",
#     "kedro.framework",
#     "kedro.runner",
#     "kedro.io",
#     "kedro.pipeline",
#     "kedro.extras",
# ]:
#     logging.getLogger(name).setLevel(logging.WARNING)

# # (optional) quiet root logger too
# logging.getLogger().setLevel(logging.WARNING)

In [27]:
companies_selection = companies_summary.query(
    "(n_assets < 100) & (n_countries > 1) & (n_assets > 8) & (sector == 'Power')"
    ).company_id.tolist()


# companies_selection = None
# [
#     # multinational megacorps
#     # "CP_7876088876044165226",
#     # "CN_9186444779649860568",
#     "CN_8600108312240451561",
#     "CP_1512176126791706747",
#     # big greentech owners
#     "CN_6660639238798673502",
#     "CN_6660639238798673502",
#     "CN_5719632086864744401",
#     # big carbontech owners
#     "CN_8600108312240451561",
#     "CN_7548398708980274705",
#     "CN_5252218731344992786",
#     # random other owners, with 10-20 assets
#     "CN_8249155112555313068",
#     "CP_7671368023139165011",
#     "CN_7263620466430749129",
#     "CP_5133603177600074280",
#     "CP_1405113695717703383",
#     "CN_7237425157254272056",
#     # random other owners, with <10 assets
#     "CN_1325960156574879189",
#     "CN_8258543408338880789",
#     "CN_4206272115616897750",
#     "CN_413233497131578182",
#     "CP_2845696436723078206",
#     "CN_6870637186458717950",
#     "CN_1465642096900403277",
#     "CN_4903484062166566625",
#     "CP_4899418540238054262",
#     "CP_1564781859061095794",
# ]
len(companies_selection)

283

In [7]:


# Define your parameter overrides
runs_configuration = {
    "company_granularity":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": True,
        "apply_retirement":False,
        "apply_decreasing_staggered_shock":False,
    },
    "asset_granularity":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":False,
        "apply_decreasing_staggered_shock":False,
    },
    "asset_granularity_with_retirement":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":True,
        "apply_decreasing_staggered_shock":False,
    },
    "asset_granularity_with_staggered_shock":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":False,
        "apply_decreasing_staggered_shock":True,
    },
    "asset_granularity_with_staggered_shock_and_retirement":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement":True,
        "apply_decreasing_staggered_shock":True,
    }
}


In [8]:
from IPython.display import clear_output

all_late_sudden_trajectories = {}
all_staggered_shock_results = {}
all_companies_npvs = {}
all_run_params = {}

total_runs = len(runs_configuration)

for idx, (run_name, run_params) in enumerate(runs_configuration.items(), start=1):
    clear_output(wait=True)  # clears the cell output each iteration
    
    print("================================================")
    print("================================================")
    print(f"Running {run_name}...")
    print(f"Run {idx}/{total_runs}")
    print("================================================")
    print("================================================")
    
    with KedroSession.create(
        project_path=Path.cwd(),
        extra_params=run_params,
    ) as session:
        session.run(pipeline_name="__default__", tags=tags)

        run_id = uuid.uuid4()

        # late_sudden_trajectories = session.load("late_sudden_trajectories")
        late_sudden_trajectories = pd.read_csv(
            "data/07_model_output/companies_late_sudden_trajectories.csv"
        )
        late_sudden_trajectories["run_id"] = run_id
        staggered_shock_results = pd.read_csv(
            "data/07_model_output/asset_level_staggered_shock.csv"
        )
        staggered_shock_results["run_id"] = run_id
        asset_npvs = pd.read_csv(
            "data/07_model_output/asset_npv.csv"
        )
        asset_npvs["run_id"] = run_id
        company_technology_npvs = pd.read_csv(
            "data/07_model_output/company_technology_npv.csv"
        )
        company_technology_npvs["run_id"] = run_id
        companies_npvs = pd.read_csv(
            "data/07_model_output/company_npv.csv"
        )
        companies_npvs["run_id"] = run_id

        run_params_df = pd.DataFrame([run_params])
        run_params_df["run_id"] = run_id

        all_late_sudden_trajectories[run_name] = late_sudden_trajectories
        all_staggered_shock_results[run_name] = staggered_shock_results
        all_companies_npvs[run_name] = companies_npvs
        all_run_params[run_name] = run_params_df

        # Copy plot folders to {workspace_dir}/{run_name}/
        run_workspace_dir = workspace_dir / run_name
        run_workspace_dir.mkdir(parents=True, exist_ok=True)

        late_sudden_trajectories.to_csv(run_workspace_dir / "all_late_sudden_trajectories.csv", index=False)
        staggered_shock_results.to_csv(run_workspace_dir / "asset_level_staggered_shock.csv", index=False)
        asset_npvs.to_csv(run_workspace_dir / "asset_npv.csv", index=False)
        company_technology_npvs.to_csv(run_workspace_dir / "company_technology_npv.csv", index=False)
        companies_npvs.to_csv(run_workspace_dir / "company_npv.csv", index=False)
        run_params_df.to_csv(run_workspace_dir / "run_params.csv", index=False)
        
        if "reporting" in tags:
            # Copy companies_trajectories_plots
            src_trajectories = Path("data/08_reporting/companies_trajectories_plots")
            dst_trajectories = run_workspace_dir / "companies_trajectories_plots"
            if src_trajectories.exists():
                if dst_trajectories.exists():
                    shutil.rmtree(dst_trajectories)
                shutil.copytree(src_trajectories, dst_trajectories)
                print(f"Copied companies_trajectories_plots to {dst_trajectories}")
            
            # Copy companies_staggered_shock_plots  
            src_staggered = Path("data/08_reporting/companies_staggered_shock_plots")
            dst_staggered = run_workspace_dir / "companies_staggered_shock_plots"
            if src_staggered.exists():
                if dst_staggered.exists():
                    shutil.rmtree(dst_staggered)
                shutil.copytree(src_staggered, dst_staggered)
                print(f"Copied companies_staggered_shock_plots to {dst_staggered}")

            # Copy companies_staggered_shock_plots  
            src_staggered = Path("data/08_reporting/asset_financial_trajectories")
            dst_staggered = run_workspace_dir / "asset_financial_trajectories"
            if src_staggered.exists():
                if dst_staggered.exists():
                    shutil.rmtree(dst_staggered)
                shutil.copytree(src_staggered, dst_staggered)
                print(f"Copied asset_financial_trajectories to {dst_staggered}")


Running company_granularity...
Run 1/5


[09/18/25 11:39:35] INFO     Kedro project crispy-kedro                                              ]8;id=592449;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/framework/session/session.py\session.py]8;;\:]8;id=634895;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/framework/session/session.py#329\329]8;;\

[09/18/25 11:39:38] WARNING  /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/ ]8;id=571266;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py\warnings.py]8;;\:]8;id=781201;file:///usr/local/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/warnings.py#109\109]8;;\
                             python3.10/site-packages/kedro/io/data_catalog.py:165:                                
                             KedroDeprecationWarning: `DataCatalog` has been deprecated and will be                
                             replaced by `KedroDataCatalog`, in Kedro 1.0.0.Currently some                         
                             `KedroDataCatalog` APIs have been retained for compatibility with                     
                             `DataCatalog`, including the `datasets` property and the                              
                             `get_datasets`, `_get_datasets`, `add`,` list`, `add_feed_dict`, and                  
                             `shallow_copy` methods. These will be removed or replaced with updated                
                             alternatives in Kedro 1.0.0. For more details, refer to the                           
                             documentation:                                                                        
                             https://docs.kedro.org/en/stable/data/index.html#kedrodatacatalog-expe                
                             rimental-feature                                                                      
                               warnings.warn(                                                                      
                                                                                                                   

                    INFO     Using synchronous mode for loading and saving data. Use the    ]8;id=283105;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/sequential_runner.py\sequential_runner.py]8;;\:]8;id=977553;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/runner/sequential_runner.py#68\68]8;;\
                             --async flag for potential performance gains.                                         
                             https://docs.kedro.org/en/stable/nodes_and_pipelines/run_a_pip                        
                             eline.html#load-and-save-asynchronously                                               

                    INFO     Loading data from params:shock_year (MemoryDataset)...             ]8;id=868396;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=887442;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#401\401]8;;\

...

                    INFO     Loading data from increasing_tech_late_sudden_trajectories         ]8;id=743018;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=661593;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#401\401]8;;\
                             (MemoryDataset)...                                                                    

                    INFO     Loading data from assets_with_baseline_trajectory                  ]8;id=389024;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=640241;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#401\401]8;;\
                             (MemoryDataset)...                                                                    

[09/18/25 12:07:32] INFO     Loading data from params:shock_year (MemoryDataset)...             ]8;id=598740;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=524679;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#401\401]8;;\

                    INFO     Running node: stagger_increasing_technologies() ->                         ]8;id=93964;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/pipeline/node.py\node.py]8;;\:]8;id=920172;file:///Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/kedro/pipeline/node.py#367\367]8;;\

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:24                                                                                   │
│                                                                                                  │
│   21 │   │   project_path=Path.cwd(),                                                            │
│   22 │   │   extra_params=run_params,                                                            │
│   23 │   ) as session:                                                                           │
│ ❱ 24 │   │   session.run(pipeline_name="__default__", tags=tags)                                 │
│   25 │   │                                                                                       │
│   26 │   │   run_id = uuid.uuid4()                                                               │
│   27                                                                                             │
│                                                                                                  │
│ /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/k │
│ edro/framework/session/session.py:399 in run                                                     │
│                                                                                                  │
│   396 │   │   │   run_params=record_data, pipeline=filtered_pipeline, catalog=catalog            │
│   397 │   │   )                                                                                  │
│   398 │   │   try:                                                                               │
│ ❱ 399 │   │   │   run_result = runner.run(                                                       │
│   400 │   │   │   │   filtered_pipeline, catalog, hook_manager, session_id                       │
│   401 │   │   │   )                                                                              │
│   402 │   │   │   self._run_called = True                                                        │
│                                                                                                  │
│ /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/k │
│ edro/runner/runner.py:129 in run                                                                 │
│                                                                                                  │
│   126 │   │   │   │   "Asynchronous mode is enabled for loading and saving data"                 │
│   127 │   │   │   )                                                                              │
│   128 │   │                                                                                      │
│ ❱ 129 │   │   self._run(pipeline, catalog, hook_or_null_manager, session_id)  # type: ignore[a   │
│   130 │   │                                                                                      │
│   131 │   │   self._logger.info("Pipeline execution completed successfully.")                    │
│   132                                                                                            │
│                                                                                                  │
│ /Users/bertrandgallice/code/Theia-Finance-Labs/crispy-kedro/.venv/lib/python3.10/site-packages/k │
│ edro/runner/sequential_runner.py:72 in _run                                                      │
│                                                                                                  │
│   69 │   │   │   │   "Using synchronous mode for loading and saving data. Use the --async fla    │
│   70 │   │   │   │   "for potential performance gains. https://docs.kedro.org/en/stable/nodes    │
│   71 │   │   │   )                                                                               │
│ ❱ 72 │   │   super()._run(                                 